# Siamse BiLSTM + Triplet Loss Retriver Model

## Import

In [66]:
# Standard libraries
import re
import string
import random
import pickle
import os
import json
from typing import Optional, Dict
import random
import logging
from enum import Enum
from datetime import datetime, timedelta
import json
from random import choice

# Data processing
import pandas as pd
import numpy as np
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split

# Visualization
import matplotlib.pyplot as plt

# Text preprocessing & NLP
from sklearn.feature_extraction.text import TfidfVectorizer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Machine Learning
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score

# Deep Learning - TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Embedding, LSTM, Bidirectional, Dense, Dropout, Concatenate
from tensorflow.keras.optimizers import Adam
from keras.saving import register_keras_serializable
from tensorflow.keras.layers import (
    Input, Embedding, SpatialDropout1D, Bidirectional, LSTM,
    Dense, Dropout, LayerNormalization, Lambda, Concatenate,BatchNormalization
)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2


## Pre Process Data

### Load Data

In [67]:
def load_clean_csv(folder_path):
    
    data = []
    all_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]

    for file in all_files:
        file_path = os.path.join(folder_path, file)
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split(',')
                if len(parts) == 3:
                    data.append({
                        'anchor': parts[0],
                        'positive': parts[1],
                        'negative': parts[2],
                        'source_file': file
                    })
                else:
                    continue  # skip baris tidak valid

    return pd.DataFrame(data)

In [68]:
folder_path = '../dataset/triplet_dataset/'
df = load_clean_csv(folder_path)
print("Total rows in combined DataFrame:", len(df))

Total rows in combined DataFrame: 2091


## Data Cleaning

In [69]:
def clean_text(text):
    if pd.isnull(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"\n", " ", text)                  
    text = re.sub(rf"[{re.escape(string.punctuation)}]", "", text)
    text = re.sub(r"\s+", " ", text)                 
    return text.strip()

In [70]:
for col in ['anchor', 'positive', 'negative']:
    df[col] = df[col].apply(clean_text)

print("\nSample cleaned data:")
print(df.head())


Sample cleaned data:
                                              anchor  \
0                                             anchor   
1   motor mati sendiri pas lagi pelan di lampu merah   
2    mobil bunyi kletuk dari mesin pas nyala pertama   
3              ac mobil bunyi kresekkresek pas nyala   
4  shock motor depan bunyi kletek pas lelet jalannya   

                                        positive  \
0                                       positive   
1        kenapa motor mati mendadak pas berhenti   
2  kenapa mesin mobil bunyi kletuk pas distarter   
3   kenapa ac mobil ada suara aneh pas dinyalain   
4               kenapa shock depan motor berisik   

                                negative  \
0                               negative   
1  apa itu sistem pengapian konvensional   
2       bagaimana cara merawat ban motor   
3        berapa harga busi motor iridium   
4       apa fungsi sensor tps pada motor   

                                         source_file  
0  Datas

## Triplet Validation

In [71]:
def is_valid_triplet(row):
    return all([
        isinstance(row['anchor'], str) and len(row['anchor'].split()) > 3,
        isinstance(row['positive'], str) and len(row['positive'].split()) > 3,
        isinstance(row['negative'], str) and len(row['negative'].split()) > 3,
    ])

In [72]:
df = df[df.apply(is_valid_triplet, axis=1)]
df = shuffle(df).reset_index(drop=True)

print("Cleaned shape:", df.shape)

Cleaned shape: (2086, 4)


## Tokenization

### Tokenizer Definition

In [73]:
VOCAB_SIZE = 10000    
OOV_TOKEN = "<OOV>"
MAX_SEQUENCE_LENGTH = 30 

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token=OOV_TOKEN)
tokenizer.fit_on_texts(df['anchor'].tolist() + df['positive'].tolist() + df['negative'].tolist())

### Tokenize Data

In [74]:
def tokenize_and_pad(texts):
    sequences = tokenizer.texts_to_sequences(texts)
    padded = pad_sequences(sequences, maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')
    return padded

anchor_input   = tokenize_and_pad(df['anchor'].tolist())
positive_input = tokenize_and_pad(df['positive'].tolist())
negative_input = tokenize_and_pad(df['negative'].tolist())

print("Tokenization done")
print("Anchor sample:", anchor_input[0])
print("Shape of anchor input:", anchor_input.shape)

Tokenization done
Anchor sample: [12 58 13 70  2 37  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0]
Shape of anchor input: (2086, 30)


In [75]:
with open("../generated/siamese_tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [76]:
np.save("../generated/anchor_input.npy", anchor_input)
np.save("../generated/positive_input.npy", positive_input)
np.save("../generated/negative_input.npy", negative_input)

## Siamse LSTM Model

In [77]:
MAX_SEQUENCE_LENGTH = 30
VOCAB_SIZE          = 10000
EMBEDDING_DIM       = 128
LSTM_UNITS          = 32   
DENSE_UNITS         = 32
DROPOUT_RATE        = 0.3
MARGIN              = 0.5
LEARNING_RATE       = 1e-4
L2_REG              = 1e-5 

In [78]:
anchor_input = np.load("../generated/anchor_input.npy")
positive_input = np.load("../generated/positive_input.npy")
negative_input = np.load("../generated/negative_input.npy")

In [79]:
with open("../generated/siamese_tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

### Build Encoder

In [80]:
@register_keras_serializable(package="Custom")
def l2_normalize(t):
    return tf.math.l2_normalize(t, axis=1)

In [81]:
def build_shared_encoder(vocab_size, embedding_dim, seq_len, dropout_rate):
    inp = Input(shape=(seq_len,), name="input_text")

    x = Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        input_length=seq_len,
        mask_zero=True,
        embeddings_regularizer=l2(L2_REG),
        name="embedding_layer"
    )(inp)

    x = SpatialDropout1D(dropout_rate)(x)

    x = Bidirectional(
        LSTM(
            LSTM_UNITS, 
            dropout=dropout_rate, 
            recurrent_dropout=dropout_rate,
            kernel_regularizer=l2(L2_REG)
        ), 
        name="bilstm_layer"
    )(x)

    x = BatchNormalization()(x)
    x = Dense(DENSE_UNITS, activation="relu", kernel_regularizer=l2(L2_REG))(x)
    x = Dropout(dropout_rate)(x)
    x = LayerNormalization()(x)
    x = Lambda(l2_normalize, name="l2_norm")(x)

    return Model(inp, x, name="shared_encoder")

In [82]:
@register_keras_serializable(package="Custom")
def triplet_loss(y_true, y_pred, margin=0.5):
    anchor, positive, negative = tf.split(y_pred, num_or_size_splits=3, axis=1)
    pos_dist = tf.reduce_sum(tf.square(anchor - positive), axis=1)
    neg_dist = tf.reduce_sum(tf.square(anchor - negative), axis=1)
    basic_loss = pos_dist - neg_dist + margin
    loss = tf.reduce_mean(tf.maximum(basic_loss, 0.0))
    return loss

In [83]:
def build_siamese_model():
    encoder = build_shared_encoder(VOCAB_SIZE, EMBEDDING_DIM, MAX_SEQUENCE_LENGTH, DROPOUT_RATE)

    a_in = Input((MAX_SEQUENCE_LENGTH,), name="anchor_input")
    p_in = Input((MAX_SEQUENCE_LENGTH,), name="positive_input")
    n_in = Input((MAX_SEQUENCE_LENGTH,), name="negative_input")

    a_emb = encoder(a_in)
    p_emb = encoder(p_in)
    n_emb = encoder(n_in)

    merged = Concatenate(axis=1, name="merged_embeddings")([a_emb, p_emb, n_emb])
    return Model(inputs=[a_in, p_in, n_in], outputs=merged, name="siamese_model")

In [84]:
model = build_siamese_model()
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss=triplet_loss
)
model.summary()

c:\Users\Lenovo\.conda\envs\tensorflow_cpu\lib\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "siamese_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ anchor_input        │ (None, 30)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positive_input      │ (None, 30)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ negative_input      │ (None, 30)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_encoder      │ (None, 32)        │  1,323,616 │ anchor_input[0][… │
│ (Functional)        │                   │            │ positive_input[0… │
│                     │                   │            │ negative_input[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ merged_embeddings   │ (None, 96)        │          0 │ shared_encoder[0… │
│ (Concatenate)       │                   │            │ shared_encoder[1… │
│                     │                   │            │ shared_encoder[2… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,323,616 (5.05 MB)

 Trainable params: 1,323,488 (5.05 MB)

 Non-trainable params: 128 (512.00 B)

In [85]:
callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, min_delta=1e-4, restore_best_weights=True),
    ModelCheckpoint("../generated/siamese_best.keras", save_best_only=True, monitor="val_loss"),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6)
]

In [86]:
idx = np.arange(anchor_input.shape[0])
train_idx, val_idx = train_test_split(idx, test_size=0.1, random_state=42, shuffle=True)

X_train = {
    'anchor_input':   anchor_input[train_idx],
    'positive_input': positive_input[train_idx],
    'negative_input': negative_input[train_idx],
}
y_train = np.zeros((train_idx.shape[0], 1))

X_val = {
    'anchor_input':   anchor_input[val_idx],
    'positive_input': positive_input[val_idx],
    'negative_input': negative_input[val_idx],
}
y_val = np.zeros((val_idx.shape[0], 1))

In [87]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    batch_size=128,
    epochs=30,
    shuffle=True,
    callbacks=callbacks
)

Epoch 1/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 23s 307ms/step - loss: 0.4151 - val_loss: 0.0891 - learning_rate: 1.0000e-04
Epoch 2/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 140ms/step - loss: 0.3891 - val_loss: 0.0647 - learning_rate: 1.0000e-04
Epoch 3/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 143ms/step - loss: 0.3377 - val_loss: 0.0498 - learning_rate: 1.0000e-04
Epoch 4/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 141ms/step - loss: 0.2934 - val_loss: 0.0398 - learning_rate: 1.0000e-04
Epoch 5/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 139ms/step - loss: 0.2726 - val_loss: 0.0312 - learning_rate: 1.0000e-04
Epoch 6/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 144ms/step - loss: 0.2260 - val_loss: 0.0266 - learning_rate: 1.0000e-04
Epoch 7/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 137ms/step - loss: 0.1969 - val_loss: 0.0225 - learning_rate: 1.0000e-04
Epoch 8/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 145ms/step - loss: 0.1748 - val_loss: 0.0201 - learning_rate: 1.0000e-04
Epoch 9/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 140ms/step - loss: 0.1639 - val_loss: 0.0186 -

In [96]:
# Save full model (with loss config)
model.save("../generated/siamese_model.keras", save_format="keras")
# Save just encoder for inference
encoder = model.get_layer("shared_encoder")
encoder.save("../generated/siamese_encoder_only.keras")

In [89]:
anchor_embed = encoder.predict(anchor_input)
positive_embed = encoder.predict(positive_input)
negative_embed = encoder.predict(negative_input)

66/66 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step


In [90]:
sim_pos = np.array([cosine_similarity([anchor_embed[i]], [positive_embed[i]])[0][0] for i in range(len(anchor_embed))])
sim_neg = np.array([cosine_similarity([anchor_embed[i]], [negative_embed[i]])[0][0] for i in range(len(anchor_embed))])

In [91]:
y_true = np.ones(len(anchor_embed))
y_pred = sim_pos > sim_neg  # True if similarity to positive > negative
accuracy = accuracy_score(y_true, y_pred)

In [92]:
def siamese_inference(anchor_text, positive_text, negative_text):
    # Tokenisasi dan padding
    anchor_seq = tokenizer.texts_to_sequences([anchor_text])
    positive_seq = tokenizer.texts_to_sequences([positive_text])
    negative_seq = tokenizer.texts_to_sequences([negative_text])

    anchor_pad = pad_sequences(anchor_seq, maxlen=MAX_SEQUENCE_LENGTH, padding='post')
    positive_pad = pad_sequences(positive_seq, maxlen=MAX_SEQUENCE_LENGTH, padding='post')
    negative_pad = pad_sequences(negative_seq, maxlen=MAX_SEQUENCE_LENGTH, padding='post')

    # Dapatkan embedding dari encoder
    anchor_emb = encoder.predict(anchor_pad)
    positive_emb = encoder.predict(positive_pad)
    negative_emb = encoder.predict(negative_pad)

    # Hitung similarity
    sim_pos = cosine_similarity(anchor_emb, positive_emb)[0][0]
    sim_neg = cosine_similarity(anchor_emb, negative_emb)[0][0]

    return {
        "sim_pos": sim_pos,
        "sim_neg": sim_neg,
        "is_positive_closer": sim_pos > sim_neg
    }

result = siamese_inference(
    "saya ingin booking servis mobil besok",
    "bisa booking servis rutin mobil besok?",
    "berapa biaya ganti oli?"
)
print(result)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
{'sim_pos': 0.93243384, 'sim_neg': 0.004034847, 'is_positive_closer': True}
